In [2]:
!git clone https://github.com/ShathaTm/LK-Hadith-Corpus.git

fatal: destination path 'LK-Hadith-Corpus' already exists and is not an empty directory.


In [3]:
path = "./LK-Hadith-Corpus"

In [4]:
import glob
path = './LK-Hadith-Corpus'
files = glob.glob(path + '/**/*.csv', recursive=True)
print(files)

['./LK-Hadith-Corpus\\AbuDaud\\Chapter1.csv', './LK-Hadith-Corpus\\AbuDaud\\Chapter10.csv', './LK-Hadith-Corpus\\AbuDaud\\Chapter11.csv', './LK-Hadith-Corpus\\AbuDaud\\Chapter12.csv', './LK-Hadith-Corpus\\AbuDaud\\Chapter13.csv', './LK-Hadith-Corpus\\AbuDaud\\Chapter14.csv', './LK-Hadith-Corpus\\AbuDaud\\Chapter15.csv', './LK-Hadith-Corpus\\AbuDaud\\Chapter16.csv', './LK-Hadith-Corpus\\AbuDaud\\Chapter17.csv', './LK-Hadith-Corpus\\AbuDaud\\Chapter18.csv', './LK-Hadith-Corpus\\AbuDaud\\Chapter19.csv', './LK-Hadith-Corpus\\AbuDaud\\Chapter2.csv', './LK-Hadith-Corpus\\AbuDaud\\Chapter20.csv', './LK-Hadith-Corpus\\AbuDaud\\Chapter21.csv', './LK-Hadith-Corpus\\AbuDaud\\Chapter22.csv', './LK-Hadith-Corpus\\AbuDaud\\Chapter23.csv', './LK-Hadith-Corpus\\AbuDaud\\Chapter24.csv', './LK-Hadith-Corpus\\AbuDaud\\Chapter25.csv', './LK-Hadith-Corpus\\AbuDaud\\Chapter26.csv', './LK-Hadith-Corpus\\AbuDaud\\Chapter27.csv', './LK-Hadith-Corpus\\AbuDaud\\Chapter28.csv', './LK-Hadith-Corpus\\AbuDaud\\Chapt

In [5]:
import re
def clean_text(text):
  text = str(text).lower()
  text = re.sub(r'[a-zA-Z0-9\s]', '', text)
  return text

In [6]:
columns = [
    'Chapter_Number', 'Chapter_English', 'Chapter_Arabic',
    'Section_Number', 'Section_English', 'Section_Arabic',
    'Hadith_Number', 'English_Hadith', 'English_Isnad',
    'English_Matn', 'Arabic_Hadith', 'Arabic_Isnad',
    'Arabic_Matn', 'Arabic_Grade', 'English_Grade'
]
columns.append('Cleaned_Hadith')

In [7]:
import pandas as pd

all_hadith = []
for file in files:
  df = pd.read_csv(file, names=columns, skiprows=1)
  df['Cleaned_Hadith'] = df['English_Hadith'].astype(str).apply(clean_text)
  all_hadith.extend(df[columns].values.tolist())

In [8]:
hadith_df = pd.DataFrame(all_hadith, columns=columns)
# hadith_df.head()
hadith_df.to_csv('cleaned_hadith.csv', index=False)

In [9]:
!pip install sentence-transformers

In [10]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [11]:
embeddings = model.encode(hadith_df['Cleaned_Hadith'].values)

In [12]:
import numpy as np
embeddings = np.array(embeddings)

In [13]:
np.save('hadith_embeddings.npy', embeddings)
embeddings = np.load('hadith_embeddings.npy')


In [14]:
!pip install faiss-cpu

In [15]:
import faiss

dimensions = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimensions)

faiss_index.add(embeddings)
faiss.write_index(faiss_index, "faiss_index.index")

In [16]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def get_similar_hadith(query, count=5, model=model, faiss_index=faiss_index):
  query_embeddings = model.encode([query])
  distance, indices = faiss_index.search(query_embeddings, count)

  for i in range(count):
    print(hadith_df['Hadith_Number'].iloc[indices[0][i]])
    print(hadith_df['English_Hadith'].iloc[indices[0][i]])

In [17]:
get_similar_hadith("Ali")

3813
Abu Hurairah, may Allah be pleased with him, narrated that the Messenger of Allah (ﷺ) said: “O Allah, benefit me with that which You have taught me, and teach me that which will benefit me, and increase me in knowledge. All praise is due to Allah in every condition, and I seek refuge in Allah from the condition of the people of the Fire (Allāhummanfa`nī bimā `allamtanī wa `allimnī mā yanfa`unī, wa zidnī `ilma, al-ḥamdulillāhi `alā kulli ḥālin, wa a`ūdhu billāhi min ḥāli ahlin-nār).”
10
Muhammad bin ul-Muthannā narrated to us, he said Abd ur-Rahman narrated to us, he said Sufyān narrated to us, on authority of Abī Ishāq, on authority of Abīl-Ahwas, on authority of Abd Illah, he said: ‘It is enough of a lie for a man that he narrates everything he hears’.
3665
Bilal bin Yahya bin Talhah bin `Ubaidullah narrated : from his father, from his grandfather Talhah bin `Ubaidullah that when the Prophet (ﷺ) would see a crescent moon, he would say: “O Allah, bring it over us with blessing and

In [18]:
get_similar_hadith("Dajjal")

418
It was narrated from Rabi' bint Mu'awwidh bin 'Afra' that: The Messenger of Allah performed ablution washing each part three times.
2095
It was narrated from 'Ubaidullah bin 'Abdullah bin 'Utabah that 'Abdullah bin 'Abbas used to say: "The Messenger of Allah was the most generous of people, and he was most generous in Ramadan when Jibril met him. Jibril use to meet him every night during the month of Ramadan and study Quran with him." And he said: "When Jibril met him, the Messenger of Allah was more generous in doing good than the blowing wind."
1644
Mua'ada 'Adawiyya reported 'A'isha as saying: The Messenger of Allah (ﷺ) used to observe four rak'ahs in the forenoon prayer and he sometimes observed more as Allah pleased.
3615
Abu Hurairah (ra) narrated that : the Messenger of Allah (ﷺ) said: “When one of you leaves his bed then returns to it, then let him brush it off with the edge of his Izar three times, for indeed, he does not know what succeeded him upon it after him. When he 